# 🫀 Heart Disease Prediction - Problem notebook

This notebook walks through a complete machine learning pipeline to predict the presence of heart disease based on patient data.

You will perform:
- Exploratory Data Analysis (EDA)
- Data Cleaning and Preprocessing
- Encoding of categorical variables
- ML model training and evaluation

**Context**

Cardiovascular diseases (CVDs) are the number 1 cause of death globally, taking an estimated 17.9 million lives each year, which accounts for 31% of all deaths worldwide. Four out of 5 CVD deaths are due to heart attacks and strokes, and one-third of these deaths occur prematurely in people under 70 years of age. Heart failure is a common event caused by CVDs and this dataset contains 11 features that can be used to predict a possible heart disease.

**Attribute Information**

`Age`: age of the patient [years]

`Sex`: sex of the patient [M: Male, F: Female]

`ChestPainType`: chest pain type [TA: Typical Angina, ATA: Atypical Angina, NAP: Non-Anginal Pain, ASY: Asymptomatic]

`RestingBP`: resting blood pressure [mm Hg]

`Cholesterol`: serum cholesterol [mg/dl]

`FastingBS`: fasting blood sugar [1: if FastingBS > 120 mg/dl, 0: otherwise]

`RestingECG`: resting electrocardiogram results [Normal: Normal, ST: having ST-T wave abnormality, LVH: showing probable or definite left ventricular hypertrophy]

`MaxHR`: maximum heart rate achieved [Numeric value between 60 and 202]

`ExerciseAngina`: exercise-induced angina [Y: Yes, N: No]

`Oldpeak`: oldpeak = ST [Numeric value measured in depression]

`ST_Slope`: the slope of the peak exercise ST segment [Up: upsloping, Flat: flat, Down: downsloping]

**Target column:** `HeartDisease` (1 = has disease, 0 = no disease)

Source: https://www.kaggle.com/datasets/fedesoriano/heart-failure-prediction

## 📦 Imports


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, confusion_matrix, ConfusionMatrixDisplay,
    classification_report, RocCurveDisplay, PrecisionRecallDisplay
)
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import make_pipeline

sns.set_theme(style='whitegrid')
print('Imports OK')

## 📥 Load the Data


You can alternatively use colab's [import file feature](https://colab.research.google.com/notebooks/io.ipynb).

In [ ]:
df = pd.read_csv('heart_failure.csv')
df.head()

## 🧾 Initial Exploration


❗**Check the dataset info and look for potential issues. How many numerical values are there? How many categorical values?**
 Hint: use the pandas dataframe [describe](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.describe.html) or [info](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.info.html) functions.

In [ ]:
print(f'Dataset shape: {df.shape}')
# Your code here ....

In [ ]:
# Your code here ...

❗ ❗**What is the prevalence of heart disease?**
(Hint: use seaborns  countplot: https://seaborn.pydata.org/generated/seaborn.countplot.html

Alternatively you can use the histplot method directly: https://seaborn.pydata.org/generated/seaborn.histplot.html.

In [ ]:
def cat_summary(dataframe, col_name, plot=False):
    print(pd.DataFrame({col_name: dataframe[col_name].value_counts(),
                        'Ratio': 100 * dataframe[col_name].value_counts() / len(dataframe)}))
    print('##########################################')
    if plot:
        sns.countplot(x=dataframe[col_name], data=dataframe)
        plt.show()

# Your code here...

❗ **Are patients diagnosed with heart disease older or younger than the rest?**
(You can use the function below. Or you  write your own script directly)

In [ ]:
def target_summary_with_num(dataframe, target, numerical_col):
    print(dataframe.groupby(target).agg({numerical_col: 'mean'}), end='\n\n\n')

# Your code here...

## 📊 EDA: Numerical Features

❗ **Summarize the numerical features. Plot the distribution of each feature. Do you notice anything bizarre?**

(Check out the histplot seaborn method: https://seaborn.pydata.org/generated/seaborn.histplot.html)

Tryout the pairplot method from seaborn also!

Note: `FastingBS` is a binary flag (0/1), not a continuous variable — its histogram will look different from the rest.

In [ ]:
num_features = ['Age', 'RestingBP', 'Cholesterol', 'MaxHR', 'Oldpeak', 'FastingBS']

for col in num_features:
    plt.figure(figsize=(6, 4))
    # Your code here... hint: use sns.histplot with kde



## 🕵️‍♂️ Checking for Missing or Invalid Values

❗ **How many missing values are there?**
You will need to identify which columns contain missing or incorrect values. Hint: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.isna.html

In [ ]:
# Your code here...

❗ **How many zero values are there?**

In [ ]:
# Your code here...

## ⚠️ Data Cleaning: Replace Invalid Zeros

Let's treat 0 in `RestingBP` and `Cholesterol` as missing values (NaN).

In [ ]:
df['RestingBP'] = df['RestingBP'].replace(0, np.nan)
df['Cholesterol'] = df['Cholesterol'].replace(0, np.nan)

print('NaN counts after replacing invalid zeros:')
df.isna().sum()

## 🩺 Imputing Missing Values

❗ **How would you handle missing data?**

The simplest option is to fill every missing value with the column's median. But cholesterol and blood pressure aren't the same for everyone: they tend to vary between groups of patients.

🤔 **Which variables could help us make a better guess for a patient's missing value?**

Hint: Use the [fillna](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.fillna.html) method.
Some strategies: https://towardsdatascience.com/8-methods-for-handling-missing-values-with-python-pandas-842544cdf891


In [ ]:
# Your code here...

In [ ]:
# Check your dataframe
print('Remaining NaN values after imputation:')
print(df.isna().sum())

In [ ]:
# Re-plot numerical distributions after imputation
for col in ['Age', 'RestingBP', 'Cholesterol', 'MaxHR', 'Oldpeak']:
    plt.figure(figsize=(6, 4))
    sns.histplot(# Your code here...)
    plt.title(f'Distribution of {col} (after imputation)')
    plt.show()

❗ **Plot the correlation matrix between the different numerical features (target included). Which feature is highest correlated with the target?**

In [ ]:
f, ax = plt.subplots(figsize=(9, 6))

# Compute correlation on numeric columns only and plot heatmap
num_df = df.select_dtypes(include=[np.number])
sns.heatmap(# Your code here...)

ax.set_title('Correlation Matrix', fontsize=20)
plt.tight_layout()
plt.show()

## 📈 EDA: Categorical Features vs HeartDisease

❗ **How many samples are there in each category for each feature?**

❗ **How do the categorical variables relate to the target?**

You can also check out:  https://seaborn.pydata.org/generated/seaborn.countplot.html

In [ ]:
# Define the categorical feature list
cat_features = ['Sex', 'ChestPainType', 'RestingECG', 'ExerciseAngina', 'ST_Slope']

for col in cat_features:
    plt.figure(figsize=(6, 4))
    # hue='HeartDisease' shows breakdown by target for each category
    sns.countplot(# Your code here...)

## 🧹 Preprocessing: One-Hot Encoding Categorical Features

❗ **Encode the categorical attributes**

`pd.get_dummies` converts each category value into its own 0/1 column. We use `drop_first=False` to keep all columns explicit (helpful for interpretability), but note that `drop_first=True` avoids perfect multicollinearity for linear models.

See: https://pandas.pydata.org/docs/reference/api/pandas.get_dummies.html

In [ ]:
# Your code here...

## 🔁 Correlation with Categorical Features
❗ Now that the categorical features are encoded, recompute the correlation with the target. Does your answer from before change? Why were these features missing earlier?

In [ ]:
# Your code here...

## ✂️ Train-Test Split

Here the dataset is split into train (to build the model) and test (for evaluation).

In [ ]:
X = df_encoded.drop('HeartDisease', axis=1)
y = df_encoded['HeartDisease']

X_train, X_test, y_train, y_test = # Your code here...

print(f'Training samples: {X_train.shape[0]}')
print(f'Test samples:     {X_test.shape[0]}')

## 🔢 Logistic Regression (Baseline Model)

❗ **Train a ML model!** First, try out a [logistic regression](https://www.google.com/url?q=https%3A%2F%2Fscikit-learn.org%2Fstable%2Fmodules%2Flinear_model.html%23logistic-regression). What accuracy do you get?

See: https://scikit-learn.org/stable/modules/linear_model.html

In [ ]:
# Your code here...

### ⚖ Evaluation metrics

❗ **Plot the results. What other metrics than accuracy can you think of?**

Check out: https://scikit-learn.org/stable/api/sklearn.metrics.html

In a clinical setting, **recall (sensitivity)** is especially important, because missing a true case of heart disease is far more dangerous than a false alarm.

In [ ]:
# Your code here...

## 🌲 Other Classifiers

❗ **Compare different types of classifiers.**

Try an SVM and a Random Forest.
Check out the [sklearn classifiers](https://www.google.com/url?q=https%3A%2F%2Fscikit-learn.org%2Fstable%2Fsupervised_learning.html).

See: https://scikit-learn.org/stable/supervised_learning.html

In [ ]:
# Your code here...

In [ ]:
# Compare ROC curves for all three models

# Your code here...

## (Optional) Feature Standardisation

❗ **Why do we need feature standardisation?**
Hint: use a [Scaler object from sklearn](https://scikit-learn.org/stable/modules/preprocessing.html)


**Important rule**: the scaler must be **fit on training data only**, then applied to both train and test. Fitting on all data leaks test-set information into the scaling parameters.

In [ ]:
num_features_to_scale = ['Age', 'RestingBP', 'Cholesterol', 'MaxHR', 'Oldpeak']

scaler = StandardScaler()

# Work on copies so original X_train / X_test are preserved for cross-validation later
X_train_scaled = X_train.copy()
X_test_scaled  = X_test.copy()

# fit on TRAIN only, then transform both sets (no data leakage)
X_train_scaled[num_features_to_scale] = scaler.fit_transform(# Your code here...)
X_test_scaled[num_features_to_scale]  = scaler.transform(# Your code here...)

# Logistic Regression on scaled data
logreg_scaled = LogisticRegression(max_iter=1000, random_state=17)
logreg_scaled.fit(# Your code here...)
y_pred_lr_scaled = logreg_scaled.predict(# Your code here...)
print('Logistic Regression (scaled):')
print(f'  Accuracy: {accuracy_score(# Your code here...):.4f}')
print(classification_report(# Your code here...))

# SVM on scaled data
svm_scaled = CalibratedClassifierCV(SVC(random_state=17), cv=3)
svm_scaled.fit(# Your code here...)
y_pred_svm_scaled = svm_scaled.predict(# Your code here...)
print('SVM (scaled):')
print(f'  Accuracy: {accuracy_score(# Your code here...):.4f}')
print(classification_report(# Your code here...))

# Random Forest
rf_scaled = RandomForestClassifier(n_estimators=100, random_state=17)
rf_scaled.fit(# Your code here...)
print('Random Forest (scaled):')
print(confusion_matrix(# Your code here...))
print(classification_report(# Your code here...))

## (Optional) Robust Evaluation via Cross-Validation

❗ **Evaluate your model's performance on multiple train-test splits. What is the average performance of your model?**

A single train-test split can be lucky or unlucky, even with stratified splits. Cross-validation (CV) repeats the evaluation across `k` different splits and averages the result. This gives a much more reliable performance estimate. We report the **mean** (expected performance) and the **standard deviation** (how much it varies between splits).

⚠️ **Careful with scaling:** the scaler must be fit on the training folds only, separately in each split. A [`Pipeline`](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.make_pipeline.html) combines a scaler and model together, so `cross_val_score` handles this automatically.



In [ ]:
# Hint: 5-fold cross-validation across all three model types.
# Scale-sensitive models (LogReg, SVM) are wrapped in a pipeline: in each split,
# the scaler is fit on the training folds only, then applied to the validation fold.
models = {
    'Logistic Regression': make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=17),  # trees don't need scaling
    'SVM':                 make_pipeline(StandardScaler(), SVC()),
}

print('5-Fold Cross-Validation Results (full dataset):')
print('-' * 50)
for name, clf in models.items():
    for metric in ['accuracy', 'roc_auc', 'f1']:
        scores = cross_val_score(# Your code here...)
        print(f'{name:25s} | {metric:10s}: {scores.mean():.4f} ± {scores.std():.4f}')
    print()

## 📣 Discussion

**What is the problem you want to solve? Why is it important?**


**Are there any challenges with the data?**


**How do we avoid data leakage?**


**Is there any data leakage in this example?**


**What are the challenges of deploying this model in a clinical setting?**
